In [1]:
import pandas as pd 
folder = "/mnt/c/Users/Ultra7/TEJ__data_daily"
date = '20260323'

In [2]:
from FinMind.data import DataLoader
from src.config import FINMIND_TOKEN
finmind_api = DataLoader()
finmind_api.login_by_token(api_token=FINMIND_TOKEN)
taiwan_stock_info_df = finmind_api.taiwan_stock_info()
taiwan_stock_info_df.head()

2026-03-24 11:07:03.002 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-24 11:07:03.045 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-24 11:07:03.046 | INFO     | FinMind.data.finmind_api:get_data:153 - download TaiwanStockInfo, data_id: 


,industry_category,stock_id,stock_name,type,date
0,電子零組件業,5481,新華,tpex,2020-06-03
1,光電業,3629,地心引力,tpex,2020-06-03
2,文化創意業,3687,歐買尬,tpex,2020-06-03
3,電腦及週邊設備業,5450,寶聯通,tpex,2020-06-03
4,其他電子類,6238,勝麗,tpex,2020-06-13


In [3]:
# 將 date 轉換為 yyyy-mm-dd 格式
date_yyyy_mm_dd = pd.to_datetime(date, format='%Y%m%d').strftime('%Y-%m-%d')

# 先根據日期和市場篩選出資料（type 不是 'emerging'）
filtered_tmp = taiwan_stock_info_df[
    (taiwan_stock_info_df['date'] == date_yyyy_mm_dd) &
    (taiwan_stock_info_df['type'] != 'emerging')
]

# 再找出 industry_category 為 '創新板股票' 的 stock_id
xinban_stock_ids = set(filtered_tmp.loc[filtered_tmp['industry_category'] == '創新板股票', 'stock_id'])

# 最後排除 stock_id 為創新板股票的資料
filtered_df = filtered_tmp[~filtered_tmp['stock_id'].isin(list(xinban_stock_ids))]
filtered_df

,industry_category,stock_id,stock_name,type,date


In [4]:
# taiwan_stock_info_df[taiwan_stock_info_df['stock_id'] == '9103']

In [5]:
# filtered_df[filtered_df['stock_id']=='9103']

In [6]:
import os
def unzip_and_delete_file(date_dir):
    import zipfile
    # 解壓縮檔案到與 zip 同一個目錄
    zip_file_path = os.path.join(f"{date_dir}.zip")

    if not  os.path.exists(zip_file_path):
        print(f"壓縮檔案 {zip_file_path} 不存在，無需解壓縮。")
        return

    with zipfile.ZipFile(zip_file_path, "r") as zf:
        zf.extractall(date_dir)
        print(f"已解壓縮 {zip_file_path} 到 {date_dir}")
    # 刪除 zip_file_path 指定的壓縮檔案
    if os.path.exists(zip_file_path):
        os.remove(zip_file_path)
        print(f"已刪除壓縮檔案 {zip_file_path}")
    else:
        print(f"壓縮檔案 {zip_file_path} 不存在，無需刪除。")

In [7]:


# 組合欲讀取的資料夾路徑
OHLCV_dir = os.path.join(folder, "OHLCV")
OHLCV_date_dir = os.path.join(OHLCV_dir, date)
unzip_and_delete_file(OHLCV_date_dir)
# 取得資料夾內所有 .txt 檔案
txt_files = [f for f in os.listdir(OHLCV_date_dir) if f.endswith('.txt')]



# 若需個別讀取檔案內容，可如此處理：

for fname in txt_files:
    file_path = os.path.join(OHLCV_date_dir, fname)
    try:
        # 嘗試以 pandas 讀取（假設為 tabular 格式）
        df = pd.read_csv(file_path, sep='\t', encoding='cp950', na_values=['na', 'NA'])
    except Exception:
        # 若非標準表格檔，如需僅讀內容
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        df = content  # 或直接存字串

        
dividend_dir = os.path.join(folder, "dividend")
dividend_date_dir = os.path.join(dividend_dir, date)
unzip_and_delete_file(dividend_date_dir)
dividend_files = [f for f in os.listdir(dividend_date_dir) if f.endswith('.txt')]


# 若需個別讀取檔案內容，可如此處理：

for fname in dividend_files:
    file_path = os.path.join(dividend_date_dir, fname)
    try:
        # 嘗試以 pandas 讀取（假設為 tabular 格式）
        df_dividend = pd.read_csv(file_path, sep='\t', encoding='cp950', na_values=['na', 'NA'])
    except Exception:
        # 若非標準表格檔，如需僅讀內容
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        df_dividend = content  # 或直接存字串
display(df_dividend)
process_div = False
if not df_dividend.empty:
    print('合併dividend')
    process_div = True
    df = pd.merge(df, df_dividend, on=['證券代碼', '年月日'], how='left')
    # 修正後的寫法：直接賦值回去
    df['除息(權)參考價(元)'] = df['除息(權)參考價(元)'].fillna(1)
    df['前一日收盤價'] = df['前一日收盤價'].fillna(1)
    
# 先將"證券代碼"欄位以空格分割，取前面的數字
df['證券代碼'] = df['證券代碼'].astype(str).str.split(' ').str[0]

# 篩選1000~9999的數字字串，其他資料不要
codemask = df['證券代碼'].str.match(r'^[1-9]\d{3}$')
df = df[codemask].copy()

# 檢核：df 與 filtered_df 的「四碼股票代碼集合」是否同步
if 'filtered_df' in globals():
    if '證券代碼' in filtered_df.columns:
        filtered_codes_raw = filtered_df['證券代碼'].astype(str)
    elif 'stock_id' in filtered_df.columns:
        filtered_codes_raw = filtered_df['stock_id'].astype(str)
    else:
        raise ValueError("filtered_df 缺少 '證券代碼' 或 'stock_id' 欄位，無法比對代碼集合")

    filtered_codemask = filtered_codes_raw.str.match(r'^[1-9]\d{3}$')
    df_codes = set(df['證券代碼'].astype(str).unique())
    filtered_codes = set(filtered_codes_raw.loc[filtered_codemask].unique())

    only_in_df = sorted(df_codes - filtered_codes)
    only_in_filtered = sorted(filtered_codes - df_codes)

    if only_in_filtered:
        # 第二層檢核：若差異只存在於 filtered_df 且數量很小，嘗試從 finMind 複查
        if len(only_in_filtered) < 10:
            if 'finmind_api' not in globals():
                raise ValueError("缺少 finmind_api 物件，無法對 only_in_filtered 代碼做 finMind 複查")
            if 'date_yyyy_mm_dd' not in globals():
                raise ValueError("缺少 date_yyyy_mm_dd 變數，無法抓當天資料")

            kept_codes = []
            for stk in only_in_filtered:
                check_df = finmind_api.taiwan_stock_daily(
                    stock_id=str(stk),
                    start_date=date_yyyy_mm_dd,
                    end_date=date_yyyy_mm_dd,
                )
                if check_df is None or getattr(check_df, 'empty', False):
                    continue

                kept_codes.append(stk)

            only_in_filtered = sorted(set(kept_codes))

        if only_in_filtered:
            raise ValueError(
                "filtered_df 存在但 df 缺失的四碼股票代碼\n"
                f"只在 filtered_df 出現: {only_in_filtered[:20]}{' ...' if len(only_in_filtered) > 20 else ''}"
            )

# 可以選擇把多餘的欄位丟掉，只保留原本的 證券代碼 欄位
# df.head()
df['is_disposed'] = 0
df.loc[df['處置股票(D)'] == 'D', 'is_disposed'] = 1
df.loc[df['全額交割(Y)'] == 'Y', 'is_disposed'] = 1

壓縮檔案 /mnt/c/Users/Ultra7/TEJ__data_daily/OHLCV/20260323.zip 不存在，無需解壓縮。
壓縮檔案 /mnt/c/Users/Ultra7/TEJ__data_daily/dividend/20260323.zip 不存在，無需解壓縮。


,證券代碼,年月日,盈餘分派_迄日,息值(元),除息(權)參考價(元),前一日收盤價
0,00836B 永豐10年A公司債,20260323,20260331,0.375,28.99,29.36
1,00856B 永豐1-3年美公債,20260323,20260331,0.335,37.97,38.30
2,00857B 永豐20年美公債,20260323,20260331,0.250,23.78,24.03
3,00930 永豐ESG低碳高息,20260323,20260430,0.150,17.62,17.77
4,00958B 永豐ESG銀行債15+,20260323,20260228,0.044,9.34,9.38
5,4163 鐿鈦,20260323,20251231,3.500,104.50,108.00


合併dividend


In [8]:
# only_in_filtered

In [9]:
# for col in df.columns:
#     print(col)

### 同步產業/上市別對照表
將當日資料中出現的 **上市別**、**TEJ子產業名**、**TEJ產業名** 與 `setting_files` 內的三個 master map 同步；若有新值則自動加入並寫入 JSON（與 `rebuilde_qlib.py` 的 `update_master_map` 邏輯一致）。

In [10]:
# 與 setting_files 內產業/上市別對照表同步：當日 df 若有新產業或新上市別則寫入對照表
import json
from pathlib import Path

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", "."))
if not (PROJECT_ROOT / "setting_files").exists():
    PROJECT_ROOT = Path.cwd()
setting_dir = PROJECT_ROOT / "setting_files"

def update_master_map_sync(df, col_name, mapping_file):
    """
    與 rebuilde_qlib.update_master_map 相同邏輯：讀取對照表，將 df 中該欄位出現的新值加入對照表並存檔。
    若有新值則分配下一個整數編號並寫入 JSON。
    """
    if col_name not in df.columns:
        return None
    if mapping_file.exists():
        with open(mapping_file, "r", encoding="utf-8") as f:
            master_map = json.load(f)
    else:
        master_map = {}
    unique_values = df[col_name].fillna("Unknown").astype(str).unique()
    changed = False
    for val in unique_values:
        if val not in master_map:
            master_map[val] = len(master_map)
            changed = True
    if changed:
        mapping_file.parent.mkdir(parents=True, exist_ok=True)
        with open(mapping_file, "w", encoding="utf-8") as f:
            json.dump(master_map, f, ensure_ascii=False, indent=4)
        print(f"已更新對照表: {mapping_file.name}")
    return master_map

# TEJ 原始欄位 → 對照表檔案（與 rebuilde_qlib 一致）
mapping_configs = {
    "exchange": ("上市別", setting_dir / "exchange_master_map.json"),
    "subindustry": ("TEJ子產業名", setting_dir / "subindustry_master_map.json"),
    "industry": ("TSE產業名", setting_dir / "industry_master_map.json"),
}
for name, (col, path) in mapping_configs.items():
    master_map = update_master_map_sync(df, col, path)
    # 將字串對應成代碼並寫入 df（與 rebuilde_qlib 一致，後續 dump 需要數值）
    if master_map is not None:
        df[name] = df[col].fillna("Unknown").astype(str).map(master_map).astype(float)

In [11]:
# TEJ 產業 → TEJ 子產業 對應關係：存成 JSON，並檢查是否與既有映射一致（有變更則提醒手動處理）
import json
from pathlib import Path

INDUSTRY_COL = "TEJ產業名"
SUBINDUSTRY_COL = "TEJ子產業名"
TEJ_MAPPING_PATH = setting_dir / "tej_industry_to_subindustry.json"

if INDUSTRY_COL not in df.columns or SUBINDUSTRY_COL not in df.columns:
    print(f" 跳過：df 缺少「{INDUSTRY_COL}」或「{SUBINDUSTRY_COL}」")
else:
    # 從當日 df 建出：每個 TEJ 產業 → 該產業下出現過的所有 TEJ 子產業（排序、唯一）
    sub = df[SUBINDUSTRY_COL].fillna("Unknown").astype(str)
    current_map = df.assign(_sub=sub).groupby(INDUSTRY_COL)["_sub"].apply(
        lambda s: sorted(s.unique().tolist())
    ).to_dict()

    if TEJ_MAPPING_PATH.exists():
        with open(TEJ_MAPPING_PATH, "r", encoding="utf-8") as f:
            saved_map = json.load(f)
    else:
        saved_map = {}

    changed_industries = []
    for ind, sub_list in current_map.items():
        if ind not in saved_map:
            saved_map[ind] = sub_list
            print(f" 新增產業映射: {ind} -> {len(sub_list)} 個子產業")
        else:
            old_set = set(saved_map[ind])
            new_set = set(sub_list)
            if old_set != new_set:
                added = new_set - old_set
                removed = old_set - new_set
                changed_industries.append({
                    "industry": ind,
                    "added": sorted(added),
                    "removed": sorted(removed),
                    "old": saved_map[ind],
                    "new": sub_list,
                })

    if changed_industries:
        print("\n⚠️ 以下產業的「子產業」與既有映射不一致，請手動檢查並更新 JSON：")
        for item in changed_industries:
            print(f"  產業: {item['industry']}")
            if item["added"]:
                print(f"    本次新增子產業: {item['added']}")
            if item["removed"]:
                print(f"    本次消失子產業: {item['removed']}")
            print(f"    既有: {item['old']}")
            print(f"    當日: {item['new']}")
        print("  請手動編輯 setting_files/tej_industry_to_subindustry.json 後再執行本 cell 確認。")

    # 寫入映射（新產業已加入 saved_map；有變更的產業保留舊值，由使用者手動改）
    TEJ_MAPPING_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(TEJ_MAPPING_PATH, "w", encoding="utf-8") as f:
        json.dump(saved_map, f, ensure_ascii=False, indent=2)
    print(f" 已寫入 TEJ 產業↔子產業映射: {TEJ_MAPPING_PATH.name}（共 {len(saved_map)} 個產業）")

 已寫入 TEJ 產業↔子產業映射: tej_industry_to_subindustry.json（共 96 個產業）


### 減資：第一天只存待處理，恢復交易日才寫入 df
- **第一天**（`股票股利上市日` 不為空）：只計算恢復交易日與兩個復權因子，寫入 `setting_files/capital_reduction_pending.json`，**不寫回 df**（因停止交易期間無日資料）。
- **恢復交易日**（股票股利上市日當天）：從 pending 讀出該批股票，將 `adj_factor_volume`（成交量調整因子）、`adj_factor_price`（價格調整因子）寫入當日 df，並自 pending 移除。
- **公式**：成交量復權 = `1/(1 - 減資率%/100)`；價格調整 = `前日收盤價（元）/ 除權參考價（元）`。

In [12]:
# 減資：第一天只存待處理，恢復交易日（股票股利上市日）當天才寫入 df
import numpy as np
from pathlib import Path

pending_path = Path(os.getenv("PROJECT_ROOT", ".")) / "setting_files" / "capital_reduction_pending.json"
if not pending_path.parent.exists():
    pending_path = Path.cwd() / "setting_files" / "capital_reduction_pending.json"
pending_path.parent.mkdir(parents=True, exist_ok=True)

# 僅當「股票股利上市日」有值且「減資率 %」不為 0 時才認定為減資
mask_cap_red = df["股票股利上市日"].notna() & (df["減資率 %"].astype(float).fillna(0) != 0)
today_dt = pd.to_datetime(date, format="%Y%m%d", errors="coerce")

# --- 1) 本日為減資第一天：只計算並存入 pending，不寫回 df ---
if mask_cap_red.any():
    next_td = pd.to_datetime(
        df.loc[mask_cap_red, "股票股利上市日"].astype(float).astype("Int64").astype(str),
        format="%Y%m%d", errors="coerce"
    )
    rate = df.loc[mask_cap_red, "減資率 %"].fillna(0)
    denom = (1 - rate / 100).replace(0, np.nan)
    vol_factor = (1 / denom).values
    prev_close = df.loc[mask_cap_red, "前日收盤價（元）"].astype(float)
    ex_ref = df.loc[mask_cap_red, "除權參考價（元）"].astype(float).replace(0, np.nan)
    price_factor = (prev_close / ex_ref).values
    new_rows = pd.DataFrame({
        "instrument": df.loc[mask_cap_red, "證券代碼"].astype(str).values,
        "next_trading_day": next_td.values,
        "capital_reduction_volume_adj_factor": vol_factor,
        "capital_reduction_price_adj_factor": price_factor,
    })
    if pending_path.exists():
        try:
            pending = pd.read_json(pending_path, convert_dates=["next_trading_day"])
            pending = pd.concat([pending, new_rows], ignore_index=True)
        except (ValueError, pd.errors.EmptyDataError):
            pending = new_rows.copy()
    else:
        pending = new_rows.copy()
    pending.to_json(pending_path, orient="records", date_format="iso", indent=2, force_ascii=False)
    print("本日減資程序啟動第一天，已寫入待處理:", mask_cap_red.sum(), "筆")

# --- 2) 每天檢查 capital_reduction_pending.json：若 next_trading_day 已到，將調整因子寫入 df ---
df["adj_factor_volume"] = 1
df["adj_factor_price"] = 1
if pending_path.exists():
    try:
        pending = pd.read_json(pending_path, convert_dates=["next_trading_day"])
    except (ValueError, pd.errors.EmptyDataError):
        pending = pd.DataFrame(columns=["instrument", "next_trading_day", "capital_reduction_volume_adj_factor", "capital_reduction_price_adj_factor"])
    if not pending.empty:
        match = pending["next_trading_day"].dt.normalize() == today_dt.normalize()
        if match.any():
            to_apply = pending.loc[match]
            for _, row in to_apply.iterrows():
                # 證券代碼在 df 為字串；JSON 讀回 instrument 可能為 int，需一致化再比對
                inst = str(row["instrument"])
                m = df["證券代碼"] == inst
                if m.any():
                    df.loc[m, "adj_factor_volume"] = row["capital_reduction_volume_adj_factor"]
                    df.loc[m, "adj_factor_price"] = row["capital_reduction_price_adj_factor"]
                else:
                    print("警告：pending 中 instrument", inst, "不在本日 df，略過寫入")
            pending = pending.loc[~match]
            pending.to_json(pending_path, orient="records", date_format="iso", indent=2, force_ascii=False)
            print("本日為恢復交易日，已寫入 adj_factor_volume/adj_factor_price 並自待處理移除:", to_apply["instrument"].tolist())

In [13]:
if process_div:
    print('處理除息')
    df['adj_price_tmp' ] = df['前一日收盤價'] / df['除息(權)參考價(元)']
    df['adj_factor_price'] = df['adj_factor_price'] * df['adj_price_tmp']
    print(df[~(df['adj_factor_price']==1)]['證券代碼'].unique())

處理除息
['4163']


In [14]:
# row_2330 = df[df['證券代碼'] == '7716']

# # row_2330 = df[df['instrument'] == '4174']
# if not row_2330.empty:
#     for col in row_2330.columns:
#         print(f"{col}: {row_2330.iloc[0][col]}")
# else:
#     print("找不到證券代碼為 2330 的資料")

In [15]:
# 找到開盤價是na的資料
volume_zero_rows = df[(df['成交量(千股)']==0) & (df['成交值(千元)']==0)]
volume_zero_stock_id = volume_zero_rows['證券代碼'].unique().tolist()
display(volume_zero_stock_id)

['2073',
 '2321',
 '2924',
 '3454',
 '3629',
 '4413',
 '5345',
 '5455',
 '6597',
 '6846',
 '7716',
 '8077']

In [16]:
untradable_stock_id = ['3454']
for stock_id in untradable_stock_id:
    if stock_id not in volume_zero_stock_id:
        raise ValueError(f"找不到 {stock_id} 的資料")
print(len(untradable_stock_id))

1


In [17]:
# df_4714 = df[df['instrument'] == '4174']
# for col in df_4714.columns:
#     print(f"{col}: {df_4714[col].values}")

In [18]:
df['is_tradable'] = 1
if '證券代碼' in df.columns:
    df.loc[df['證券代碼'].isin(untradable_stock_id), 'is_tradable'] = 0
else:
    df.loc[df['instrument'].isin(untradable_stock_id), 'is_tradable'] = 0
print(df['is_tradable'].value_counts())

is_tradable
1    1927
0       1
Name: count, dtype: int64


In [19]:
rename_dict = {
    # --- 直接 rename（1 對 1）---
    '證券代碼': 'instrument',
    '年月日': 'date',
    '開盤價(元)': 'open',
    '最高價(元)': 'high',
    '最低價(元)': 'low',
    '收盤價(元)': 'close',
    '成交量(千股)': 'volume',
    '成交值(千元)': 'amount',
    '報酬率％': 'return_day',
    '週轉率％': 'turnover',
    '市值(百萬元)': 'market_value_million',
    '最後揭示買價': 'last_bid',
    '最後揭示賣價': 'last_ask',
    '報酬率-Ln': 'log_return_day',
    '市值比重％': 'market_value_pct',
    '成交值比重％': 'trading_value_pct',
    '成交筆數(筆)': 'numtrades',
    '本益比-TSE': 'pe_ratio_tse',
    '本益比-TEJ': 'pe_ratio_tej',
    '股價淨值比-TSE': 'pb_ratio_tse',
    '股價淨值比-TEJ': 'pb_ratio_tej',
    '股價營收比-TEJ': 'ps_Ratio_tej',
    '股利殖利率-TSE': 'div_yield_tse',
    '高低價差%': 'spread',
    # --- 需要額外處理（非單純 rename）---
    # '上市別'            → exchange        (需搭配變更市場歷史做時間序列 mapping)
    # 'TEJ子產業名'       → subindustry     (需搭配 TEJ子產業變更歷史做時間序列 mapping)
    # 'TSE產業名'         → industry        (需搭配 TSE產業變更歷史做時間序列 mapping)
    # adj_factor_price    → 由除權息資料計算得出
    # adj_factor_volume   → 由除權息資料計算得出
    # is_tradable         → 由全額交割、處置等多欄位綜合判斷
}


In [20]:
save_path = f"/home/newt/quant_project/TEJ_processed_data/{date}"

if not os.path.exists(save_path):
    os.makedirs(save_path)
save_col = ['is_tradable', 'is_disposed' , 'exchange', 'subindustry', 'industry' , 'adj_factor_price', 'adj_factor_volume']
save_col =  save_col + list(rename_dict.values())
df.rename(columns=rename_dict, inplace=True)
df = df[save_col]
for col in df.columns:
    if col not in save_col:
        print(col)

In [21]:
display(df.head())

from datetime import datetime
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

,is_tradable,is_disposed,exchange,subindustry,industry,adj_factor_price,adj_factor_volume,instrument,date,open,...,market_value_pct,trading_value_pct,numtrades,pe_ratio_tse,pe_ratio_tej,pb_ratio_tse,pb_ratio_tej,ps_Ratio_tej,div_yield_tse,spread
355,1,0,1.0,257.0,12.0,1.0,1,1101,20260323,23.15,...,0.161,0.130,15026,NaN,NaN,0.76,0.7676,1.0829,3.51,2.3355
358,1,0,1.0,257.0,12.0,1.0,1,1102,20260323,34.15,...,0.114,0.053,4810,12.08,11.5853,0.67,0.6970,1.6583,6.73,1.5805
359,1,0,1.0,257.0,12.0,1.0,1,1103,20260323,13.40,...,0.010,0.001,462,16.07,19.3081,0.49,0.5854,3.5362,4.07,1.0989
360,1,0,1.0,257.0,12.0,1.0,1,1104,20260323,29.10,...,0.019,0.003,769,12.43,12.4546,0.86,0.8636,2.5500,6.34,2.0548
361,1,0,1.0,257.0,12.0,1.0,1,1108,20260323,14.85,...,0.006,0.000,118,6.32,6.3488,1.05,1.0529,1.2644,6.76,1.0067


2026-03-24 11:07:03


In [22]:
na_instruments = df[df.isna().any(axis=1)]['instrument'].tolist()
print(len(na_instruments))
print(na_instruments)

629
['1101', '1213', '1235', '1236', '1301', '1303', '1304', '1305', '1308', '1309', '1310', '1312', '1313', '1314', '1316', '1321', '1324', '1325', '1326', '1337', '1338', '1340', '1417', '1418', '1432', '1435', '1438', '1440', '1441', '1444', '1445', '1447', '1449', '1452', '1454', '1455', '1457', '1460', '1465', '1466', '1467', '1471', '1512', '1515', '1516', '1517', '1526', '1528', '1530', '1531', '1536', '1538', '1569', '1583', '1584', '1586', '1589', '1591', '1595', '1599', '1617', '1626', '1710', '1711', '1721', '1742', '1762', '1781', '1799', '1802', '1808', '1809', '1813', '1905', '1906', '1909', '2002', '2007', '2014', '2017', '2022', '2023', '2024', '2025', '2034', '2035', '2038', '2061', '2064', '2067', '2073', '2101', '2102', '2104', '2208', '2231', '2236', '2239', '2241', '2314', '2316', '2321', '2331', '2332', '2337', '2338', '2340', '2342', '2344', '2349', '2369', '2371', '2380', '2388', '2390', '2392', '2399', '2401', '2405', '2406', '2408', '2409', '2413', '2417', '24

In [23]:
import random

def na_instruments_na_cols_generator(df, instrument_list):
    """
    生成器：針對 instrument_list 中的每個 instrument，列出為 NA 的欄位
    """
    shuffled = instrument_list.copy()
    random.shuffle(shuffled)
    for instrument_id in shuffled:
        row = df[df['instrument'] == instrument_id]
        if row.empty:
            yield f"{instrument_id} 不存在於資料中"
        else:
            na_cols = row.columns[row.isna().any()].tolist()
            yield f"{instrument_id} 的 NA 欄位: {na_cols}"

# 每次運行此 cell 可獲得不同順序的結果
gen = na_instruments_na_cols_generator(df, na_instruments)

# # 範例：每次執行 next(gen) 可以檢查一個 instrument
# # 若要一次看全部：
# for result in gen:
#     print(result)
# 檢查是不是 ohlcv 欄位都不是 na
ohlcv_cols = ['open', 'high', 'low', 'close', 'volume' , 'adj_factor_price' , 'adj_factor_volume' , 'amount' ,'market_value_million' , 'is_tradable' , 'is_disposed' , 'exchange' , 'subindustry' , 'industry' ]
ohlcv_not_na = df[ohlcv_cols].notna().all(axis=1)
print(f"全部 ohlcv 欄位都不是 NA 的 row 數量: {ohlcv_not_na.sum()} / {len(df)}")

# 若想顯示這些 rows
# print(df[ohlcv_not_na])

全部 ohlcv 欄位都不是 NA 的 row 數量: 1928 / 1928


In [24]:
# import shutil
# df.to_parquet(os.path.join(save_path, f"{date}.parquet"), index=False)


# OHLCV_processed_folder = os.path.join(OHLCV_dir, "processed")
# dividend_processed_folder = os.path.join(dividend_dir, "processed")
# if not os.path.exists(OHLCV_processed_folder) or not os.path.exists(dividend_processed_folder):

#     raise FileNotFoundError(f"目標資料夾{OHLCV_processed_folder} 或 {dividend_processed_folder} 不存在")
# try:
#     shutil.move(OHLCV_date_dir, OHLCV_processed_folder)
#     print(f"{OHLCV_date_dir} 已移動到 {OHLCV_processed_folder}")
# except Exception as e:
#     print(f"移動過程中發生錯誤: {e}")
# try:
#     shutil.move(dividend_date_dir, dividend_processed_folder)
#     print(f"{dividend_date_dir} 已移動到 {dividend_processed_folder}")
# except Exception as e:
#     print(f"移動過程中發生錯誤: {e}")